# Explainability, Fairness, Robustness, ML Engineering Hygiene and the Interview Bank

Notebook for **Chapter 16** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch16.html

Tip: change a number, predict the output, then run the cell again.

## 16.1 Explainability: Global Methods

In [ ]:
base = roc_auc_score(y[te], rf.predict_proba(X[te])[:, 1]); prng = np.random.default_rng(1)
for j, name in enumerate(names):
    drops = []
    for r in range(20):
        Xp = X[te].copy(); Xp[:, j] = prng.permutation(Xp[:, j])
        drops.append(base - roc_auc_score(y[te], rf.predict_proba(Xp)[:, 1]))
    print(name, round(np.mean(drops), 4), round(np.std(drops), 4))
pi = permutation_importance(rf, X[te], y[te], scoring="roc_auc", n_repeats=20, random_state=1)

## 16.2 Explainability: Local Methods (Shapley values, LIME, Counterfactuals)

In [ ]:
import shap
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(X_test)      # additive contributions, one per feature per row
shap.summary_plot(sv, X_test)

## 16.3 Fairness and Ethics

## 16.4 Robustness and Security

## 16.5 ML Engineering Hygiene

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline
# WRONG: pick the 20 "best" columns using ALL rows, then cross-validate
Xsel = SelectKBest(f_classif, k=20).fit_transform(Xn, yn)
wrong = cross_val_score(LogisticRegression(max_iter=1000), Xsel, yn, cv=KFold(5, shuffle=True, random_state=0)).mean()
# RIGHT: selection inside the pipeline, so it is re-done on each training fold
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
right = cross_val_score(pipe, Xn, yn, cv=KFold(5, shuffle=True, random_state=0)).mean()

## 16.6 The From-Scratch Implementations List

In [ ]:
dz2 = (p - y) / len(X); dW2 = h.T @ dz2; db2 = dz2.sum(0)
dh  = dz2 @ W2.T;       dz1 = dh * (1 - h**2); dW1 = X.T @ dz1; db1 = dz1.sum(0)

0 loss 0.6712
500 loss 0.0051
5000 loss 0.0005
predictions [0.    0.999 0.999 0.001]
grad check analytic -6.468e-05 numeric -6.468e-05


In [ ]:
def kfold(n, k, seed=0):
    idx = np.random.default_rng(seed).permutation(n); folds = np.array_split(idx, k)
    for i in range(k):
        yield np.concatenate([folds[j] for j in range(k) if j != i]), folds[i]

## 16.7 The Interview Bank

Answer these aloud, in simple words, in under a minute each. The model answers below are short on purpose.

## 16.8 Three Specialist Problems: Survival, Uplift and Ranking

Three kinds of question look like classification or regression but break their rules. "How long until the customer leaves?" has people who have not left yet. "Who should get the coupon?" is about the *change* the coupon causes, not who buys. "Which ten results do we show?" cares about order, not about each score. Each needs its own metric and data design.

In [ ]:
import numpy as np
# Kaplan-Meier by hand: times, event=1 observed, 0 censored
t = np.array([2, 3, 3, 5, 6, 8, 8, 9, 12, 12]); e = np.array([1, 1, 0, 1, 0, 1, 1, 0, 1, 0])
S = 1.0; print("time at_risk deaths  S(t)")
for u in np.unique(t[e == 1]):
    n = (t >= u).sum(); d = ((t == u) & (e == 1)).sum(); S *= 1 - d / n; print(f"{u:4d} {n:7d} {d:6d}  {S:.4f}")
# naive: drop censored vs treating censored as events
print("naive share of ALL 10 rows with an event by t=8:", ((t <= 8) & (e == 1)).sum() / len(t))
print("KM estimate of P(event by t=8):", round(1 - 0.4114, 4))

time at_risk deaths  S(t)
   2      10      1  0.9000
   3       9      1  0.8000
   5       7      1  0.6857
   8       5      2  0.4114
  12       2      1  0.2057
naive share of ALL 10 rows with an event by t=8: 0.5
KM estimate of P(event by t=8): 0.5886


In [ ]:
import numpy as np
rng = np.random.default_rng(0); n = 40000
x = rng.normal(size=n); T = rng.integers(0, 2, n)
tau = np.where(x > 0, 0.10, -0.04)                      # persuadable if x>0, sleeping-dog otherwise
p = 0.10 + tau * T; y = rng.random(n) < p
from sklearn.ensemble import GradientBoostingClassifier as G
X = x[:, None]; m1 = G(max_depth=2, n_estimators=50, random_state=0).fit(X[T==1], y[T==1]); m0 = G(max_depth=2, n_estimators=50, random_state=0).fit(X[T==0], y[T==0])
up = m1.predict_proba(X)[:, 1] - m0.predict_proba(X)[:, 1]
print("true mean effect", round(tau.mean(), 3), " mean predicted uplift", up.mean().round(3))
order = np.argsort(-up)
for q in (0.25, 0.5, 1.0):
    k = order[: int(q * n)]; a = y[k][T[k]==1].mean() - y[k][T[k]==0].mean()
    print(f"top {int(q*100)}% by uplift: observed effect {a:.3f}")
r = np.argsort(-(m0.predict_proba(X)[:,1]))   # wrong: target by response propensity
k = r[: n // 4]; print("top 25% by baseline response: effect", round(y[k][T[k]==1].mean() - y[k][T[k]==0].mean(), 3))

true mean effect 0.03  mean predicted uplift 0.024
top 25% by uplift: observed effect 0.120
top 50% by uplift: observed effect 0.096
top 100% by uplift: observed effect 0.024
top 25% by baseline response: effect 0.004


In [ ]:
import numpy as np
def dcg(r): return sum((2**v - 1) / np.log2(i + 2) for i, v in enumerate(r))
rel = [3, 2, 3, 0, 1, 2]
print("DCG@6", round(dcg(rel), 4), "ideal", round(dcg(sorted(rel, reverse=True)), 4), "NDCG", round(dcg(rel) / dcg(sorted(rel, reverse=True)), 4))
from sklearn.metrics import ndcg_score
print("sklearn", round(ndcg_score([rel], [[6,5,4,3,2,1]]), 4))

DCG@6 13.8483 ideal 14.5954 NDCG 0.9488
sklearn 0.9608


## Chapter recap

## Practice set

**1. (easy)** A model is `f = 3·x₁ + 0·x₂`. Which feature has zero permutation importance? **Answer:** `x₂`, because shuffling it does not change any prediction.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch16.html